# SCAG Demo — ACDC mid-SAX 2-D

Step-by-step Semantic Coactivation Graph analysis for **ACDC mid-SAX 2-D**.

Input policy: **the deterministic central SAX slice at ED and ES per patient**.

The notebook exposes the full analysis rather than hiding it inside `run_model()`:

**data → trained model → activations → X → R → Q → R–Q alignment → graph → communities → semantic analyses → causal ablation**


## 1. Imports

In [1]:
import numpy as np
import torch
from collections import Counter
from IPython.display import display

from scag.config import SCAGConfig
from scag.data import (
    load_datasets,
    build_graph_analysis_cohort,
    make_split_safe,
    build_loaders,
    phase_encode_segments,
)
from scag.models import build_model
from scag.activations import extract_activations
from scag.semantics import compute_neuron_concept_matrix
from scag.matrices import compute_R, compute_Q, align_R_Q, compute_edge_reliability
from scag.graph import (
    analyze_graph,
    density_threshold,
    seed_stability,
    community_cards,
    area_coupling,
)
from scag.ablation import community_ablation
from scag.pipeline import compute_area_profiles, run_semantic_granularity

/home/kislay/anaconda3/envs/cinema/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

In [2]:
cfg = SCAGConfig(dataset="acdc")

print("Dataset:", cfg.dataset)
print("Device:", cfg.device)
print("Data root:", cfg.data_root)
print("Model root:", cfg.model_root)
print("Classes:", cfg.classes)
print("Concepts:", cfg.concept_names)

Dataset: acdc
Device: cpu
Data root: /media/kislay/New Volume/Turab/results/cinema_reproduction/processed/acdc
Model root: /media/kislay/New Volume/Turab/results/cinema_reproduction/architecture_bank/2d/acdc_sax_mid_2d
Classes: ('DCM', 'HCM', 'MINF', 'NOR', 'RV')
Concepts: ('background', 'RV_ED', 'MYO_ED', 'LV_ED', 'RV_ES', 'MYO_ES', 'LV_ES')


/home/kislay/anaconda3/envs/cinema/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


The notebook uses the full SCAG defaults. For a quick code smoke test only, you can temporarily lower the permutation/ablation counts in `cfg`; use the defaults for reported experiments.

## 3. Load the official train / validation / test splits

In [3]:
train_dataset, val_dataset, test_dataset, splits = load_datasets(cfg)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Loaded ACDC: train=90, val=10, test=50
Train: 90
Validation: 10
Test: 50


## 4. Build the graph-analysis cohort

The graph-analysis cohort is **official validation + official test**, matching the existing SCAG experiments. Therefore the official test split is not treated as an untouched test set for this analysis.

In [4]:
eval_dataset = build_graph_analysis_cohort(val_dataset, test_dataset, cfg,)
eval_dataset = make_split_safe(train_dataset,eval_dataset,)

print("Graph-analysis cohort:", len(eval_dataset))

[acdc] graph cohort: val (10) + test (50) = 60
[acdc] official test is not held out from graph analysis.
[split-safe] train=images, eval=images
[split-safe] 60 -> 60 (dropped 0)
Graph-analysis cohort: 60


## 5. Build deterministic loaders

In [5]:
train_loader, eval_loader = build_loaders(train_dataset, eval_dataset, batch_size=cfg.batch_size, num_workers=cfg.num_workers,)

print("Train batches:", len(train_loader))
print("Evaluation batches:", len(eval_loader))

Train batches: 6
Evaluation batches: 4


## 6. Load one trained architecture

In [6]:
tag = "resnet18"

model, hook_target, layer_name = build_model(tag, cfg)
model = model.to(cfg.device).eval()

print("Architecture:", tag)
print("Feature layer:", layer_name)

[acdc/resnet18] loaded checkpoint: /media/kislay/New Volume/Turab/results/cinema_reproduction/architecture_bank/2d/acdc_sax_mid_2d/resnet18/stacked/imagenet/seed_0/best_val_mcc.pt
[acdc/resnet18] SCAG layer: encoder.layer4
Architecture: resnet18
Feature layer: encoder.layer4


## 7. Extract training activations and masks

In [7]:
train_acts, train_segs = extract_activations(
    model,
    train_loader,
    hook_target,
    cfg.device,
)

print("Train activations:", tuple(train_acts.shape))
print("Train masks:", tuple(train_segs.shape))

Train activations: (90, 512, 6, 6)
Train masks: (90, 2, 192, 192)


## 8. Extract graph-cohort activations and masks

In [8]:
eval_acts, eval_segs = extract_activations(
    model,
    eval_loader,
    hook_target,
    cfg.device,
)

print("Evaluation activations:", tuple(eval_acts.shape))
print("Evaluation masks:", tuple(eval_segs.shape))

Evaluation activations: (60, 512, 6, 6)
Evaluation masks: (60, 2, 192, 192)


## 9. Convert anatomy masks to phase-aware concepts

The shared semantic vocabulary is:

- `RV_ED`, `MYO_ED`, `LV_ED`
- `RV_ES`, `MYO_ES`, `LV_ES`

with background kept as concept `0`.

In [9]:
train_segs = phase_encode_segments(train_segs)
eval_segs = phase_encode_segments(eval_segs)

print("Encoded labels:", torch.unique(train_segs).tolist())

Encoded labels: [0, 1, 2, 3, 4, 5, 6]


## 10. Compute the neuron–concept association matrix X

`X[k, c]` measures the association between neuron/filter `k` and semantic concept `c` using the SCAG dissection procedure.

In [10]:
X, labeled = compute_neuron_concept_matrix(
    train_acts,
    train_segs,
    num_concepts=cfg.num_concepts,
    alpha=cfg.alpha,
    eta=cfg.eta,
)

lab_idx = np.where(labeled)[0]

print("X shape:", X.shape)
print("Labeled neurons:", len(lab_idx), "/", X.shape[0])

  dissect 128/512
  dissect 256/512
  dissect 384/512
  dissect 512/512
X shape: (512, 7)
Labeled neurons: 96 / 512


## 11. Inspect dominant concepts

In [11]:
concept_counts = Counter(
    X[lab_idx].argmax(axis=1).tolist()
)

{
    cfg.concept_name_map[int(concept_id)]: int(count)
    for concept_id, count in sorted(concept_counts.items())
}

{'RV_ED': 5, 'LV_ED': 11, 'RV_ES': 29, 'MYO_ES': 8, 'LV_ES': 43}

## 12. Compute functional coactivation R

Each activation map is spatially pooled, then neuron–neuron Spearman correlation is computed.

In [12]:
R_train = compute_R(train_acts)
R_eval = compute_R(eval_acts)

print("R_train:", R_train.shape)
print("R_eval:", R_eval.shape)

R_train: (512, 512)
R_eval: (512, 512)


## 13. Compute semantic similarity Q

`Q` is cosine similarity between the semantic concept profiles of the **same labeled neurons**.

In [13]:
Q = compute_Q(X[lab_idx])

print("Q:", Q.shape)

Q: (96, 96)


## 14. Train–evaluation edge-weight reproducibility

In [14]:
edge_reliability = compute_edge_reliability(
    R_train,
    R_eval,
    lab_idx,
)

print(f"Spearman rho = {edge_reliability:.4f}")

Spearman rho = 0.8901


## 15. Full-matrix R–Q alignment

In [15]:
R_train_lab = R_train[np.ix_(lab_idx, lab_idx)]
R_eval_lab = R_eval[np.ix_(lab_idx, lab_idx)]

alignment_train = align_R_Q(
    R_train_lab,
    Q,
    n_perm=cfg.n_mantel_permutations,
)

alignment_eval = align_R_Q(
    R_eval_lab,
    Q,
    n_perm=cfg.n_mantel_permutations,
)

print("Train (r, z, p):", alignment_train)
print("Eval  (r, z, p):", alignment_eval)

Train (r, z, p): (0.1845280490034636, 10.737397398536029, 0.000999000999000999)
Eval  (r, z, p): (0.14121523951895337, 7.912549523004449, 0.000999000999000999)


## 16. Analyze the fixed-threshold graph

In [16]:
graph_eval, artifacts_eval = analyze_graph(
    R_eval,
    X,
    labeled,
    tau=cfg.tau,
    n_perm=cfg.n_battery_permutations,
)

graph_eval

{'n_nodes': 512,
 'n_edges': 35976,
 'n_isolates': 0,
 'n_comm': 4,
 'modularity': 0.45180527588132974,
 'purity': 0.5937391152908394,
 'purity_z': 4.367695632192646,
 'purity_p': 0.003992015968063872,
 'ami': 0.12435640277944945,
 'ami_z': 5.678982177346109,
 'ami_p': 0.001996007984031936,
 'assort': 0.09934226295293666,
 'assort_z': 8.27121529578337,
 'assort_p': 0.001996007984031936,
 'soft_delta': 0.044592183914531014,
 'soft_z': 2.9193155101846266,
 'soft_p': 0.013972055888223553,
 'alignment_tau_r': 0.10220422765514442,
 'alignment_tau_z': 5.650569222028287,
 'alignment_tau_p': 0.003992015968063872}

## 17. Inspect Louvain communities

In [17]:
communities = artifacts_eval["communities"]

print("Number of communities:", len(communities))
for community_id, community in enumerate(communities):
    print(f"Community {community_id:02d}: {len(community)} neurons")

Number of communities: 4
Community 00: 162 neurons
Community 01: 155 neurons
Community 02: 131 neurons
Community 03: 64 neurons


## 18. Analyze a density-matched graph

In [18]:
tau_dm = density_threshold(
    R_eval,
    cfg.graph_density,
)

graph_dm, artifacts_dm = analyze_graph(
    R_eval,
    X,
    labeled,
    tau=tau_dm,
    n_perm=cfg.n_battery_permutations,
)

print("Density-matched threshold:", tau_dm)
print("Edges:", graph_dm["n_edges"])
print("Modularity:", graph_dm["modularity"])

Density-matched threshold: 0.8971380939149767
Edges: 2618
Modularity: 0.7266772609304519


## 19. Compare semantic granularity

The **same neurons, R matrix, edge weights, and thresholds** are interpreted at three semantic levels:

1. anatomy,
2. cardiac phase,
3. anatomy × phase.

In [19]:
granularity_df, granularity_summary, semantic_views = run_semantic_granularity(
    tag=tag,
    R_train=R_train,
    R_eval=R_eval,
    X=X,
    labeled=labeled,
    tau_dm=tau_dm,
    cfg=cfg,
)

display(granularity_df)


------------------------------------------------------------------------
SEMANTIC LEVEL: ANATOMY
------------------------------------------------------------------------
Dominant concepts: {'RV': 34, 'MYO': 8, 'LV': 54}
  comm 00: RV[anatomy]x16, MYO[anatomy]x4, LV[anatomy]x9
  comm 01: LV[anatomy]x23, RV[anatomy]x10, MYO[anatomy]x3
  comm 02: LV[anatomy]x14, RV[anatomy]x7, MYO[anatomy]x1
  comm 03: LV[anatomy]x8, RV[anatomy]x1

------------------------------------------------------------------------
SEMANTIC LEVEL: PHASE
------------------------------------------------------------------------
Dominant concepts: {'ED': 4, 'ES': 92}
  comm 00: ES[phase]x28, ED[phase]x1
  comm 01: ED[phase]x3, ES[phase]x33
  comm 02: ES[phase]x22
  comm 03: ES[phase]x9

------------------------------------------------------------------------
SEMANTIC LEVEL: ANATOMY_PHASE
------------------------------------------------------------------------
Dominant concepts: {'RV_ED': 5, 'LV_ED': 11, 'RV_ES': 29, 'MY

,tag,dataset,semantic_level,n_labeled,n_effective_classes,dominant_counts,alignment_train_r,alignment_train_z,alignment_train_p,alignment_eval_r,...,eval_assort_z,eval_soft_delta,eval_soft_z,eval_alignment_tau_r,eval_alignment_tau_z,eval_modularity,dm_ami_z,dm_assort_z,dm_soft_z,dm_modularity
0,resnet18,acdc,anatomy,96,3,"{""RV"": 34, ""MYO"": 8, ""LV"": 54}",0.175326,10.258831,0.000999,0.131852,...,3.753050,0.039778,2.569882,0.094090,5.370908,0.451805,4.719199,3.429874,3.118003,0.726677
1,resnet18,acdc,phase,96,2,"{""ED"": 4, ""ES"": 92}",0.025926,1.323691,0.093906,0.014584,...,0.064591,-0.000496,-1.605299,-0.000531,-0.031176,0.451805,0.888836,-0.292001,-2.198180,0.726677
2,resnet18,acdc,anatomy_phase,96,5,"{""RV_ED"": 5, ""LV_ED"": 11, ""RV_ES"": 29, ""MYO_ES...",0.184528,10.737397,0.000999,0.141215,...,8.271215,0.044592,2.919316,0.102204,5.650569,0.451805,5.493555,5.966637,3.279571,0.726677


## 20. Louvain seed stability

In [20]:
ami_mean, ami_sd, partition_stability = seed_stability(
    artifacts_eval["G"],
    artifacts_eval["lab_nodes"],
    artifacts_eval["labels"],
    n_seeds=cfg.n_louvain_seeds,
)

print("AMI vs semantics:", ami_mean, "+/-", ami_sd)
print("Partition self-stability:", partition_stability)

  seed  0: AMI=0.1437 z_null=+6.5
  seed  1: AMI=0.1591 z_null=+7.6
  seed  2: AMI=0.1591 z_null=+7.7
  seed  3: AMI=0.1437 z_null=+7.3
  seed  4: AMI=0.1513 z_null=+6.9
  seed  5: AMI=0.1458 z_null=+6.7
  seed  6: AMI=0.1437 z_null=+8.2
  seed  7: AMI=0.1513 z_null=+6.3
  seed  8: AMI=0.1591 z_null=+7.3
  seed  9: AMI=0.1437 z_null=+7.0
AMI vs semantics: 0.1500419644813827 +/- 0.0065181918077158165
Partition self-stability: 0.8618048066380751


## 21. Community semantic enrichment

In [21]:
enrichment_df = community_cards(
    artifacts_eval,
    cfg.concept_name_map,
    cfg.concept_category_map,
)

display(enrichment_df)

  comm 00: RV_ES[anatomy_phase]x16, LV_ED[anatomy_phase]x6, MYO_ES[anatomy_phase]x4, LV_ES[anatomy_phase]x3
  comm 01: RV_ED[anatomy_phase]x5, LV_ES[anatomy_phase]x22, RV_ES[anatomy_phase]x5, LV_ED[anatomy_phase]x1, MYO_ES[anatomy_phase]x3
  comm 02: LV_ES[anatomy_phase]x12, RV_ES[anatomy_phase]x7, LV_ED[anatomy_phase]x2, MYO_ES[anatomy_phase]x1
  comm 03: LV_ES[anatomy_phase]x6, LV_ED[anatomy_phase]x2, RV_ES[anatomy_phase]x1


,comm,concept,k,p,p_adj
0,0,4,16,0.000677,0.010837
1,0,6,3,1.000000,1.000000
2,0,3,6,0.068184,0.272735
3,0,5,4,0.188933,0.481197
4,1,4,5,0.998824,1.000000
5,1,6,22,0.011258,0.060042
6,1,1,5,0.006168,0.049341
7,1,3,1,0.996114,1.000000
8,1,5,3,0.637994,1.000000
9,2,4,7,0.522932,0.929657


## 22. Anatomy-area coupling

In [22]:
area_profiles = compute_area_profiles(
    train_segs,
    cfg.num_concepts,
)

area_stats = area_coupling(
    R_eval,
    X,
    labeled,
    area_profiles,
    n_perm=cfg.n_cooccurrence_permutations,
)

area_stats

  positive (n= 1341): area-profile cosine=0.8792 z=+4.4 p=0.0020
      anti (n= 1035): area-profile cosine=0.8433 z=-5.6 p=0.0010


{'positive': (1341,
  0.8791620286003486,
  4.404350227776016,
  0.001998001998001998),
 'anti': (1035, 0.8432803422967807, -5.615212032586216, 0.000999000999000999)}

## 23. Causal community ablation

Each Louvain community is ablated by zeroing its feature channels at the selected layer. Its prediction effect is compared with size-matched random-neuron ablations.

**Note:** the default `n_ablation_samples=1000` is intentionally expensive for final experiments.

In [ ]:
node2label = dict(zip(
    artifacts_eval["lab_nodes"],
    artifacts_eval["labels"].tolist(),
))

ablation_df = community_ablation(
    model=model,
    hook_target=hook_target,
    loader=eval_loader,
    communities=artifacts_eval["communities"],
    num_filters=train_acts.shape[1],
    device=cfg.device,
    val_segs=eval_segs,
    node2label=node2label,
    concept_names=cfg.concept_name_map,
    n_random=cfg.n_ablation_samples,
)

display(ablation_df)

## 24. Rank communities by causal effect

In [ ]:
if ablation_df is not None and len(ablation_df):
    columns = [
        column
        for column in ["comm", "size", "kl", "rnd", "rnd_sd", "flip", "specificity", "spec_ratio"]
        if column in ablation_df.columns
    ]
    display(
        ablation_df[columns]
        .sort_values("kl", ascending=False)
        .reset_index(drop=True)
    )

## Outputs produced in this notebook

- `X`: neuron–concept association matrix
- `R_train`, `R_eval`: functional coactivation matrices
- `Q`: neuron–neuron semantic similarity matrix
- `alignment_train`, `alignment_eval`: R–Q alignment
- `artifacts_eval["G"]`: coactivation graph
- `artifacts_eval["communities"]`: Louvain communities
- `granularity_df`: anatomy vs phase vs anatomy×phase
- `enrichment_df`: community semantic enrichment
- `area_stats`: anatomical-burden coupling
- `ablation_df`: causal community-ablation results

For all configured architectures, run:

```bash
python run_scag.py --dataset acdc
```
